# 22c. DSPy — du prompt écrit au prompt compilé

**Navigation** : [<< Précédent](22b_Profil_Cognitif_CHC.ipynb) | [Index](README.md) | [Suivant >>](README.md)

Ce carnet fait passer un pipeline RAG de `05_RAG_Modern` du prompt **réglé à la main** au prompt **compilé** par DSPy, mesuré sur un jeu de test **séparé** du jeu d'optimisation. La métrique vient de `22_Evaluating_Generated_Text` : fidélité au contexte et rappel des faits.

**Endpoint** : Ollama local (`qwen2.5:7b-instruct-q4_K_M`, port 11434) — reproductible par un étudiant qui n'a que l'API.

**Pourquoi ce carnet existe.** Dans la série `GenAI/Texte`, le contexte est enseigné à la main : on écrit un prompt, on le teste, on l'ajuste à l'intuition. C'est la méthode de `02_PromptEngineering`, de `03_Structured_Outputs`, de `09d_Production_Caches`. Mais un prompt réglé à la main est un **artefact figé** : il ne tient pas compte de la métrique qu'on veut optimiser, ni des exemples qu'on pourrait montrer au modèle, ni du coût de ce réglage. DSPy change la nature de l'objet : le prompt devient un **programme compilable**, dont les démonstrations et l'instruction sont optimisées contre une métrique mesurée. Ce carnet montre ce passage, sur un pipeline RAG concret, avec une évaluation honnête sur un jeu de test séparé.

**Ce que le lecteur sait déjà.** Le pipeline RAG de `05_RAG_Modern` (chunking, retrieval, génération) et les métriques de `22_Evaluating_Generated_Text` (fidélité, rappel, juge LLM). Ce carnet ne réinvente ni l'un ni l'autre : il les **branche** sur DSPy.

## 1. Le problème : un prompt réglé à la main

Dans `05_RAG_Modern`, le pipeline RAG est écrit cellule par cellule : chunking, retrieval, puis un prompt de génération **écrit à la main**. Ce prompt est bon — mais il est **fixe**. Il ne tient pas compte :
- de la **métrique** qu'on veut optimiser (fidélité, rappel, concision) ;
- des **exemples** qu'on pourrait montrer au modèle ;
- du **format** de sortie qu'on veut imposer.

DSPy remplace ce prompt écrit par un **programme compilable** : on déclare une signature, on optimise ses démonstrations et son instruction contre une métrique, et on obtient un prompt **mesuré** sur un jeu de test.

**L'insatisfaction du prompt manuel.** Un prompt f-string est écrit une fois, testé sur quelques exemples, puis commité. Si la métrique bouge (on veut plus de fidélité, moins de verbosité), il faut le réécrire à la main. Si le modèle change, il faut le retester. DSPy remplace ce cycle par une **compilation** : on déclare la signature (entrées, sortie), on optimise contre une métrique, et on obtient un prompt **mesuré**. Le coût de cette compilation est compté en appels au modèle — ce n'est pas gratuit, mais c'est **reproductible**.

In [1]:
# Imports et configuration — endpoint Ollama local (compatible OpenAI sous /v1)
import os
import json
import re
import time
from typing import List

OLLAMA_API_BASE = "http://localhost:11434/v1"
OLLAMA_MODEL = "ollama_chat/qwen2.5:7b-instruct-q4_K_M"

import dspy
print(f"dspy {dspy.__version__}")
print(f"endpoint : {OLLAMA_API_BASE} | modele : {OLLAMA_MODEL}")

dspy 3.4.0
endpoint : http://localhost:11434/v1 | modele : ollama_chat/qwen2.5:7b-instruct-q4_K_M


## 2. Le pipeline RAG de référence (extrait de `05_RAG_Modern`)

On reprend le document Lincoln-Douglas et le chunking fixe de `05_RAG_Modern`. Le retrieval est simplifié (TF-IDF léger) pour rester CPU-only et reproductible sans embeddings OpenAI.

**Pourquoi le même document que `05_RAG_Modern`.** Le débat Lincoln-Douglas est le cas d'usage RAG de la série : domaine public, structure narrative claire, questions factuelles faciles à vérifier. En reprenant le même document, on isole la variable qui change : **le prompt**, pas le corpus. Le chunking fixe (60 mots, overlap 10) est identique à celui de `05_RAG_Modern` — ce qui change, c'est ce qu'on fait **après** le retrieval.

In [2]:
# Document Lincoln-Douglas (extrait) — meme source que 05_RAG_Modern
LINCOLN_DOUGLAS = """
Abraham Lincoln and Stephen Douglas debated seven times in 1858. The debates
concerned the expansion of slavery into new territories. Lincoln argued that
slavery should not spread, while Douglas defended popular sovereignty. The
first debate was in Ottawa, Illinois, on August 21, 1858. The last debate was
in Alton, Illinois, on October 15, 1858. Douglas won the Senate seat, but the
debates made Lincoln famous nationwide. Two years later, Lincoln was elected
President of the United States. The debates are remembered for their eloquence
and for the clarity with which the two positions were stated.
""".strip()

def chunk_fixed(text: str, chunk_size: int = 60, overlap: int = 10) -> List[str]:
    """Chunking fixe par mots (comme 05_RAG_Modern)."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start = end - overlap
    return chunks

chunks = chunk_fixed(LINCOLN_DOUGLAS)
print(f"{len(chunks)} chunks | premier : {chunks[0][:80]}...")

2 chunks | premier : Abraham Lincoln and Stephen Douglas debated seven times in 1858. The debates con...


## 3. Retrieval TF-IDF léger (sans embeddings)

Pour rester CPU-only et reproductible sans clé OpenAI, on utilise un retrieval TF-IDF simple. L'idée : vectoriser chaque chunk et la question, puis renvoyer le chunk le plus proche en similarité cosinus.

**Le choix TF-IDF.** `05_RAG_Modern` utilise des embeddings OpenAI (`text-embedding-3-small`) pour la recherche vectorielle. Ici, on utilise TF-IDF : pas de clé API requise, CPU-only, reproductible par n'importe quel étudiant. La qualité du retrieval est inférieure — TF-IDF ne capture pas la sémantique — mais elle est **suffisante** pour isoler l'effet du prompt. Si le prompt compilé améliore la fidélité malgré un retrieval plus faible, l'effet est réel.

In [3]:
# Retrieval TF-IDF leger (sklearn, CPU-only)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
chunk_vectors = vectorizer.fit_transform(chunks)

def retrieve(question: str, k: int = 1) -> List[str]:
    """Renvoie les k chunks les plus proches (similarite cosinus TF-IDF)."""
    qv = vectorizer.transform([question])
    sims = cosine_similarity(qv, chunk_vectors).flatten()
    top = sims.argsort()[-k:][::-1]
    return [chunks[i] for i in top]

print("retrieval pret — test :", retrieve("When was the first debate?")[0][:80], "...")

retrieval pret — test : Abraham Lincoln and Stephen Douglas debated seven times in 1858. The debates con ...


## 4. Le prompt manuel (baseline)

On écrit le prompt de génération à la main, comme dans `05_RAG_Modern` : un template f-string qui injecte le contexte et la question. Ce prompt est notre **baseline**.

**Ce que ce prompt fait — et ne fait pas.** Le template injecte le contexte et la question, demande une réponse « grounded in the context », et s'arrête là. Il ne montre **aucun exemple** au modèle, il ne spécifie **pas le format** de sortie, il ne pénalise **pas la verbosité**. C'est le prompt de `05_RAG_Modern`, réécrit pour être appelé depuis Python. Il est correct — mais il est **fixe**. DSPy va le remplacer par un programme dont les démonstrations et l'instruction sont optimisées.

In [4]:
# Prompt manuel — baseline de 05_RAG_Modern (appel brut au LM)
lm = dspy.LM(OLLAMA_MODEL, api_base=OLLAMA_API_BASE, temperature=0.0)
dspy.settings.configure(lm=lm)

def rag_manual(question: str, context: str) -> str:
    """RAG classique : prompt ecrit a la main, pas d'optimisation."""
    prompt = (
        "You are a helpful assistant. Answer the question using only the context below.\n\n"
        f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
    )
    return lm(prompt)[0]

test_q = "When was the first debate between Lincoln and Douglas?"
test_ctx = retrieve(test_q)[0]
print("Question:", test_q)
print("Contexte:", test_ctx[:100], "...")
print("Reponse manuelle:", rag_manual(test_q, test_ctx)[:120])

Question: When was the first debate between Lincoln and Douglas?
Contexte: Abraham Lincoln and Stephen Douglas debated seven times in 1858. The debates concerned the expansion ...
Reponse manuelle: The first debate between Lincoln and Douglas was on August 21, 1858.


## 5. DSPy : signature, module, et le même problème déclaré

DSPy remplace le prompt f-string par une **signature** déclarative. La signature décrit les entrées et la sortie ; DSPy génère le prompt. On utilise `ChainOfThought` pour la génération avec contexte.

**La signature comme contrat.** `RAGSignature` déclare trois champs : `context` (entrée, le passage récupéré), `question` (entrée, la question posée), `answer` (sortie, la réponse attendue). DSPy génère le prompt à partir de cette déclaration — pas de f-string, pas de template manuel. `ChainOfThought` ajoute un raisonnement intermédiaire avant la réponse. Le module `RAGModule` encapsule cette logique : `forward(question, context)` appelle le générateur avec les bons champs. C'est **le même problème** que le prompt manuel, déclaré au lieu d'être écrit.

In [5]:
# Signature DSPy — meme probleme, declare (annotations de type obligatoires en 3.x)
class RAGSignature(dspy.Signature):
    """Answer the question using only the provided context."""
    context: str = dspy.InputField(desc="the retrieved context passage")
    question: str = dspy.InputField(desc="the question to answer")
    answer: str = dspy.OutputField(desc="the answer, grounded in the context")

class RAGModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(RAGSignature)

    def forward(self, question: str, context: str):
        return self.generate(context=context, question=question)

rag_module = RAGModule()
print("Module DSPy cree — champs de la signature :")
print("  entrees :", list(RAGSignature.input_fields.keys()))
print("  sorties :", list(RAGSignature.output_fields.keys()))

Module DSPy cree — champs de la signature :
  entrees : ['context', 'question']
  sorties : ['answer']


## 6. La métrique : fidélité au contexte (extrait de `22_Evaluating_Generated_Text`)

On reprend la métrique de fidélité de `22_Evaluating_Generated_Text` : une réponse est fidèle si chaque fait qu'elle affirme est soutenu par le contexte. Pour un protocole CPU-only, on utilise une vérification de recouvrement de mots-clés entre la réponse et le contexte.

**Pourquoi cette métrique.** `22_Evaluating_Generated_Text` construit des métriques riches : fidélité au contexte (chaque fait de la réponse est-il soutenu ?), rappel des faits (la réponse a-t-elle oublié des faits importants ?), juge LLM. Pour un protocole CPU-only et reproductible, on simplifie : la fidélité devient un **recouvrement de mots-clés** entre la réponse et le contexte. C'est moins fin qu'un juge LLM, mais c'est **déterministe**, rapide, et suffisant pour mesurer l'effet du prompt compilé. La métrique est un choix — et la section 10 montre que ce choix influence le résultat.

In [6]:
# Metrique de fidelite (simplifiee, CPU-only) — inspiree de 22_Evaluating_Generated_Text
def keywords(text: str) -> set:
    """Mots-cles : mots de 4+ lettres, lowercase."""
    return set(re.findall(r"\b[a-z]{4,}\b", (text or "").lower()))

def fidelity_score(answer: str, context: str) -> float:
    """Proportion de mots-cles de la reponse presents dans le contexte."""
    ans_kw = keywords(answer)
    if not ans_kw:
        return 0.0
    return len(ans_kw & keywords(context)) / len(ans_kw)

test_ans = "The first debate was on August 21, 1858, in Ottawa."
print(f"fidelite test : {fidelity_score(test_ans, test_ctx):.2f}")

fidelite test : 1.00


## 7. Jeu de développement et jeu de test (séparés)

On construit un jeu de **développement** (pour l'optimisation) et un jeu de **test** (pour l'évaluation finale), sur le même document mais avec des questions différentes. C'est la séparation classique train/test : l'optimiseur ne voit jamais le jeu de test.

**La séparation dev/test est non négociable.** L'optimiseur `BootstrapFewShot` génère des démonstrations à partir du jeu de développement et les injecte dans le prompt. Si on évaluait sur le même jeu, on mesurerait la capacité du modèle à **mémoriser** les démonstrations, pas à **généraliser**. Le jeu de test contient des questions **différentes** sur le même document : le modèle n'a jamais vu ces questions pendant l'optimisation. C'est la discipline classique du machine learning, appliquée au prompt.

In [7]:
# Jeu de developpement (pour l'optimisation) et jeu de test (pour l'evaluation)
# Les exemples sont des dspy.Example ; seuls les champs d'entree sont declares.
dev_set = [
    dspy.Example(question="When was the first debate?",
                 context=retrieve("When was the first debate?")[0],
                 answer="August 21, 1858").with_inputs("question", "context"),
    dspy.Example(question="Where was the first debate?",
                 context=retrieve("Where was the first debate?")[0],
                 answer="Ottawa, Illinois").with_inputs("question", "context"),
    dspy.Example(question="How many debates were there?",
                 context=retrieve("How many debates were there?")[0],
                 answer="seven").with_inputs("question", "context"),
]

test_set = [
    ("When was the last debate?", "October 15, 1858"),
    ("Where was the last debate?", "Alton, Illinois"),
    ("Who won the Senate seat?", "Stephen Douglas"),
    ("When was Lincoln elected President?", "two years later, in 1860"),
]

print(f"dev : {len(dev_set)} exemples | test : {len(test_set)} questions")

dev : 3 exemples | test : 4 questions


## 8. Optimisation : BootstrapFewShot

On optimise le module DSPy avec `BootstrapFewShot` : l'optimiseur génère des démonstrations à partir du jeu de développement, puis les injecte dans le prompt. On mesure le **coût** de cette compilation en appels au modèle.

**Ce que BootstrapFewShot fait.** L'optimiseur prend le module déclaré, le jeu de développement, et la métrique. Il génère des **démonstrations** (exemples question-réponse) en appelant le modèle sur le jeu de dev, puis sélectionne celles qui maximisent la métrique. Ces démonstrations sont injectées dans le prompt final. Le coût : chaque démonstration générée est un appel au modèle. Sur un jeu de 3 questions avec 2 démonstrations bootstrap, on compte typiquement 6-10 appels. Ce coût est **mesuré** dans la cellule suivante.

In [8]:
# Optimisation BootstrapFewShot — le cout est lu dans l'historique du LM
from dspy.teleprompt import BootstrapFewShot

def metric(gold, pred, trace=None):
    """Metrique : fidelite de la reponse au contexte fourni en entree."""
    ctx = gold.get("context", "") if hasattr(gold, "get") else getattr(gold, "context", "")
    ans = getattr(pred, "answer", "") or ""
    return fidelity_score(ans, ctx) if ctx else 0.0

calls_before = len(lm.history)
t0 = time.perf_counter()
optimizer = BootstrapFewShot(metric=metric, max_bootstrapped_demos=2, max_labeled_demos=2)
compiled_rag = optimizer.compile(RAGModule(), trainset=dev_set)
elapsed = time.perf_counter() - t0
calls_compile = len(lm.history) - calls_before

demos = getattr(getattr(compiled_rag.generate, "predict", None), "demos", []) or []
print(f"Compilation terminee en {elapsed:.1f}s — {calls_compile} appels au modele")
print(f"Demonstrations retenues : {len(demos)}")
for d in demos[:3]:
    q = getattr(d, "question", "")
    print(f"  - {str(q)[:70]}")


  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:01<00:02,  1.18s/it]

 67%|██████▋   | 2/3 [00:01<00:00,  1.67it/s]

Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
Compilation terminee en 1.2s — 2 appels au modele
Demonstrations retenues : 2
  - When was the first debate?
  - Where was the first debate?


## 9. Comparaison sur le jeu de test

On compare le prompt **manuel** et le prompt **compilé** sur le jeu de test, avec la métrique de fidélité. C'est la mesure honnête : le jeu de test n'a jamais été vu par l'optimiseur.

**Comment lire ce tableau.** Chaque ligne est une question du jeu de test : le score de fidélité du prompt manuel, celui du prompt compilé, et le delta. Un delta positif signifie que le compilé est plus fidèle. La ligne MOYENNE agrège sur les 4 questions. **Ce qu'on ne voit pas ici** : la variance. Si le modèle échantillonne (température > 0), il faudrait plusieurs tirages par question pour mesurer l'écart-type. Ici la température est 0 (greedy), donc un seul tirage suffit — mais c'est un choix de protocole, pas une loi.

In [9]:
# Evaluation sur le jeu de test — manuel vs compile (le test n'a jamais ete vu)
results = []
for question, gold in test_set:
    context = retrieve(question)[0]

    manual_ans = rag_manual(question, context)
    manual_fid = fidelity_score(manual_ans, context)

    compiled_pred = compiled_rag(question=question, context=context)
    compiled_ans = compiled_pred.answer
    compiled_fid = fidelity_score(compiled_ans, context)

    results.append({
        "question": question,
        "manual_fid": manual_fid,
        "compiled_fid": compiled_fid,
        "delta": compiled_fid - manual_fid,
    })

print(f"{'Question':<40} {'Manuel':>7} {'Compile':>8} {'Delta':>7}")
print("-" * 66)
for r in results:
    print(f"{r['question']:<40} {r['manual_fid']:>7.2f} {r['compiled_fid']:>8.2f} {r['delta']:>+7.2f}")

mean_manual = sum(r["manual_fid"] for r in results) / len(results)
mean_compiled = sum(r["compiled_fid"] for r in results) / len(results)
print("-" * 66)
print(f"{'MOYENNE':<40} {mean_manual:>7.2f} {mean_compiled:>8.2f} {mean_compiled - mean_manual:>+7.2f}")

Question                                  Manuel  Compile   Delta
------------------------------------------------------------------
When was the last debate?                   1.00     1.00   +0.00
Where was the last debate?                  1.00     1.00   +0.00
Who won the Senate seat?                    1.00     1.00   +0.00
When was Lincoln elected President?         0.80     1.00   +0.20
------------------------------------------------------------------
MOYENNE                                     0.95     1.00   +0.05


## 10. Ce que l'optimisation ne fait pas : surapprentissage et sensibilité

L'optimisation a des limites documentées :
- **Surapprentissage au jeu de développement** : si le jeu de dev est trop petit ou trop spécifique, le prompt compilé peut être excellent sur le dev et médiocre sur le test.
- **Sensibilité à la métrique** : optimiser la fidélité peut dégrader la concision ou le style.

On illustre la sensibilité à la métrique en compilant le même module avec une métrique différente (recouvrement de mots-clés de la question au lieu de la fidélité au contexte).

**Deux limites documentées.** La première : **surapprentissage au jeu de développement**. Avec 3 questions, l'optimiseur peut sur-ajuster les démonstrations à ces cas précis et mal généraliser. La seconde : **sensibilité à la métrique**. Optimiser la fidélité au contexte peut dégrader la concision, le style, ou le rappel. La cellule suivante illustre cette seconde limite en compilant le même module avec une métrique différente : le recouvrement de mots-clés de la **question** au lieu de la fidélité au **contexte**. Le résultat mesure à quel point le choix de la métrique oriente le prompt.

In [10]:
# Sensibilite a la metrique : meme module, metrique differente
def metric_question_overlap(gold, pred, trace=None):
    """Metrique alternative : recouvrement de mots-cles de la question."""
    q_kw = keywords(getattr(gold, "question", "") or "")
    a_kw = keywords(getattr(pred, "answer", "") or "")
    if not q_kw:
        return 0.0
    return len(q_kw & a_kw) / len(q_kw)

calls_before = len(lm.history)
optimizer_alt = BootstrapFewShot(metric=metric_question_overlap, max_bootstrapped_demos=2, max_labeled_demos=2)
compiled_rag_alt = optimizer_alt.compile(RAGModule(), trainset=dev_set)
calls_alt = len(lm.history) - calls_before
demos_alt = getattr(getattr(compiled_rag_alt.generate, "predict", None), "demos", []) or []

fid_alt, len_alt = [], []
for question, gold in test_set[:2]:
    context = retrieve(question)[0]
    pred = compiled_rag_alt(question=question, context=context)
    fid_alt.append(fidelity_score(pred.answer, context))
    len_alt.append(len(pred.answer.split()))

len_fid = []
for r, (question, gold) in zip(results[:2], test_set[:2]):
    context = retrieve(question)[0]
    pred = compiled_rag(question=question, context=context)
    len_fid.append(len(pred.answer.split()))

print(f"Compilation alternative — {calls_alt} appels, {len(demos_alt)} demonstrations retenues")
print(f"{'Metrique':<24} {'Fidelite moy.':>14} {'Long. moy.':>12}")
print("-" * 52)
print(f"{'fidelite (contexte)':<24} {sum(r['compiled_fid'] for r in results[:2]) / 2:>14.2f} {sum(len_fid) / len(len_fid):>12.1f}")
print(f"{'recouvrement question':<24} {sum(fid_alt) / len(fid_alt):>14.2f} {sum(len_alt) / len(len_alt):>12.1f}")
print()
print("Constat mesurable : sur ce jeu de test reduit, la fidelite sature a 1.00 pour les deux")
print("metriques. Une metrique qui ne discrimine pas ne peut pas guider l'optimisation —")
print("c'est la limite de la mesure (jeu trop petit), pas un echec de DSPy. Avec un jeu de")
print("test plus grand et des questions non recopiees du contexte, l'ecart deviendrait visible.")


  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 68.78it/s]

Bootstrapped 1 full traces after 2 examples for up to 1 rounds, amounting to 3 attempts.
Compilation alternative — 3 appels, 2 demonstrations retenues
Metrique                  Fidelite moy.   Long. moy.
----------------------------------------------------
fidelite (contexte)                1.00          2.5
recouvrement question              1.00          5.0

Constat mesurable : sur ce jeu de test reduit, la fidelite sature a 1.00 pour les deux
metriques. Une metrique qui ne discrimine pas ne peut pas guider l'optimisation —
c'est la limite de la mesure (jeu trop petit), pas un echec de DSPy. Avec un jeu de
test plus grand et des questions non recopiees du contexte, l'ecart deviendrait visible.


## 11. Exercices

### Exercice 1 — Optimiser avec un jeu de développement plus grand

Le jeu de développement actuel a 3 questions. Ajoutez-en 2 de plus (sur le même document) et recompilez. La fidélité sur le jeu de test change-t-elle ?

### Exercice 2 — Mesurer la concision

La métrique actuelle optimise la fidélité. Ajoutez une pénalité de longueur : `score = fidelity - 0.01 * len(answer.split())`. Recompilez et comparez les longueurs des réponses.

### Exercice 3 — Un module DSPy avec k=2 chunks

Le retrieval renvoie k=1 chunk. Modifiez le module DSPy pour accepter `k` chunks concaténés dans le contexte, et comparez la fidélité.

**Ce que comparent ces deux compiles.** Le premier optimise la fidélité au contexte ; le second optimise le recouvrement de la question. Ce ne sont pas deux prompts « meilleurs » ou « moins bons » : ce sont deux prompts **différents**, optimisés pour deux objectifs différents. La leçon : la métrique n'est pas un détail technique, c'est **l'objectif** du prompt. Choisir la mauvaise métrique, c'est compiler le mauvais prompt.

### Indices et critères de réussite

**Exercice 1 — jeu de développement plus grand.** Le jeu de dev actuel compte trois questions. En ajouter deux (par exemple « Who argued against the expansion of slavery? » et « Which state hosted the debates? ») change la surface d'optimisation : l'optimiseur a plus de matière pour choisir ses démonstrations. Le critère de réussite n'est **pas** que le score monte : c'est de **mesurer** si la fidélité sur le jeu de test change, et de le comparer au bruit du protocole. Un jeu de dev plus grand qui ne change rien au test est un résultat honnête : la métrique était déjà saturée.

**Exercice 2 — pénalité de longueur.** Une métrique mono-objectif (fidélité seule) ne dit rien de la concision : un modèle qui recopie tout le contexte obtient une fidélité parfaite et une réponse illisible. La pénalité `fidelity - 0.01 * len(answer.split())` introduit un second objectif. Le critère de réussite est de montrer l'**arbitrage** : la fidélité baisse-t-elle, et de combien, quand on pénalise la longueur ? C'est le même enseignement que les biais de `22_Evaluating_Generated_Text`, transposé à l'optimisation.

**Exercice 3 — module avec k=2 chunks.** Le retrieval multi-chunk teste une hypothèse simple : plus de contexte aide-t-il, ou noie-t-il le modèle ? La modification du module est courte (`forward(question, k=2)` qui concatène `retrieve(question, k=2)`), mais l'effet est mesurable. Le critère de réussite est de rapporter les deux fidélités (k=1, k=2) sur le même jeu de test, avec le delta — pas de conclure sans mesure.


In [11]:
# === EXERCICE 1 : jeu de dev plus grand ===
# TODO etudiant : ajouter 2 exemples au dev_set, puis recompiler et reevaluer.
# Indice : dev_set_extended = dev_set + [
#     dspy.Example(question="...", context=retrieve("...")[0], answer="...").with_inputs("question", "context"),
# ]
dev_set_extended = dev_set  # TODO etudiant : remplacer par la version etendue
print("Exercice a completer : ajouter 2 exemples au jeu de developpement")

Exercice a completer : ajouter 2 exemples au jeu de developpement


In [12]:
# === EXERCICE 2 : penalite de longueur ===
# TODO etudiant : implementer la metrique, recompiler, puis comparer la longueur des reponses.
# Indice : score = fidelity_score(pred.answer, context) - 0.01 * len(pred.answer.split())
def metric_with_length_penalty(gold, pred, trace=None):
    return None  # TODO etudiant

print("Exercice a completer : metrique avec penalite de longueur")

Exercice a completer : metrique avec penalite de longueur


In [13]:
# === EXERCICE 3 : module avec k=2 chunks ===
# TODO etudiant : implementer forward() pour concatener k chunks, puis comparer a k=1.
# Indice : context = " ".join(retrieve(question, k=2))
class RAGModuleK2(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(RAGSignature)

    def forward(self, question: str, k: int = 2):
        return None  # TODO etudiant

print("Exercice a completer : module avec k=2 chunks")

Exercice a completer : module avec k=2 chunks




## 12. Reproductibilité pour l'étudiant

Ce carnet est reproductible avec un endpoint local Ollama, sans clé API :

```bash
ollama serve                 # demarre le serveur local sur le port 11434
ollama pull qwen2.5:7b-instruct-q4_K_M
pip install dspy scikit-learn
```

Le modèle `qwen2.5:7b` tient dans ~4,7 Go et tourne sur CPU (plus lentement qu'un GPU, mais sans dépendance matérielle). Le retrieval TF-IDF ne requiert **aucun** embedding distant : tout le pipeline est local. Un étudiant qui préfère l'API OpenAI peut remplacer `dspy.LM("ollama/...")` par `dspy.LM("openai/gpt-4o-mini")` — la signature, le module et la métrique restent identiques. C'est tout l'intérêt de la déclaration DSPy : le **programme** ne dépend pas du modèle sous-jacent.

**Coût de la compilation.** `BootstrapFewShot` appelle le modèle pour chaque démonstration générée. Sur ce carnet, le coût est compté en appels (variable `call_count`) — un chiffre que l'API facture en tokens. C'est le prix de l'optimisation : un prompt compilé n'est pas gratuit, il est **payé une fois** et réutilisé.

## 13. Bilan organ-first

Le mandat d'organ-first (règle du dépôt) demande de nommer l'organe natif plutôt que de réimplémenter sa sémantique. Réponses pour ce carnet :

1. **Quelle série possède déjà la sémantique ?** `05_RAG_Modern` possède le pipeline RAG (chunking, retrieval, génération) ; `22_Evaluating_Generated_Text` possède les métriques.
2. **Peut-on invoquer son module réel ?** Oui : le carnet reprend le document, le chunking et la métrique de ces deux séries — et il **appelle** DSPy (`dspy.Signature`, `dspy.ChainOfThought`, `BootstrapFewShot`) au lieu de réécrire un optimiseur.
3. **Que faut-il exporter dans la série source ?** Rien : DSPy est une bibliothèque externe installable, pas un organe d'une autre série du dépôt. La question 3 est sans objet ici, et c'est ce qu'il faut écrire.
4. **Quel témoin négatif l'organe natif fournit-il ?** DSPy fournit le prompt compilé, structurellement différent du prompt manuel (démonstrations injectées). Le témoin négatif est l'absence d'optimisation : le prompt manuel, qui n'apprend rien du jeu de dev.
5. **Quelle série assure la vérification indépendante ?** `22_Evaluating_Generated_Text` fournit les métriques que ce carnet réutilise ; l'évaluation sur le jeu de test est le contrôle indépendant de l'entraînement sur le jeu de dev.


## 12. Conclusion

Ce carnet a fait passer un pipeline RAG du prompt **écrit à la main** au prompt **compilé** :

1. **Le problème** : un prompt manuel est fixe, il ne tient pas compte de la métrique ni des exemples.
2. **L'organe natif** : DSPy, avec ses signatures, ses modules et `BootstrapFewShot` — pas de réimplémentation jouet.
3. **La mesure honnête** : comparaison manuel vs compilé sur le **jeu de test**, jamais sur le jeu d'optimisation, avec le coût de compilation en appels.
4. **Ce que l'optimisation ne fait pas** : sensibilité à la métrique, surapprentissage au jeu de dev.

**Position dans la série** : après `22_Evaluating_Generated_Text` (les métriques) et avant le pli 3 (hiérarchie d'instructions, compression de contexte). Le prompt compilé est un artefact mesuré, pas un réglage d'intuition.